<a href="https://colab.research.google.com/github/innsvi/python_for_ds_task/blob/main/Sales_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📈 Опис обраного датасету.

***E-Commerce Sales Dataset***

Датасет був обраний на kaggle.com та включає в себе понад 128 000 записів і 24 колонок. Завдяки реальним бізнес-метрикам, таким як MRP, суми оплат та статуси виконання замовлень (Fulfillment), цей датасет дає можливість провести глибокий аналіз і виявити ключові фактори впливу на прибутковість у сучасному ринковому середовищі, що робить проєкт максимально наближеним до реальних аналітичних завдань.

*Архів завантажено на гугл-диск*.


## 🔗 Завантаження датасету та попереднє ознайомлення з даними.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile
import os

In [3]:
zip_path = '/content/drive/MyDrive/data/sales_dataset.zip'
extract_path = '/content/project_data'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Файли розпаковані:", extract_path)

os.listdir(extract_path)

Файли розпаковані: /content/project_data


['Amazon Sale Report.csv',
 'P  L March 2021.csv',
 'Expense IIGF.csv',
 'International sale Report.csv',
 'May-2022.csv',
 'Cloud Warehouse Compersion Chart.csv',
 'Sale Report.csv']

In [4]:
import pandas as pd

# Завантажуємо основний звіт
amazon_df = pd.read_csv('/content/project_data/Amazon Sale Report.csv', low_memory=False)

# Завантажуємо звіт про товари
products_df = pd.read_csv('/content/project_data/Sale Report.csv')

print(f"Розмір основного датасету: {amazon_df.shape[0]} рядків, {amazon_df.shape[1]} колонок")
print(f"Розмір довідника товарів: {products_df.shape[0]} рядків, {products_df.shape[1]} колонок")

# Перевіряємо типи даних
print("\n--- Типи даних та пропуски в Amazon Sale Report ---")
amazon_df.info()

Розмір основного датасету: 128975 рядків, 24 колонок
Розмір довідника товарів: 9271 рядків, 7 колонок

--- Типи даних та пропуски в Amazon Sale Report ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  object 
 2   Date                128975 non-null  object 
 3   Status              128975 non-null  object 
 4   Fulfilment          128975 non-null  object 
 5   Sales Channel       128975 non-null  object 
 6   ship-service-level  128975 non-null  object 
 7   Style               128975 non-null  object 
 8   SKU                 128975 non-null  object 
 9   Category            128975 non-null  object 
 10  Size                128975 non-null  object 
 11  ASIN                128975 non-null  object 
 12  Courier Status      122103 no

*Перед тим як приступити до очищення даних перевіримо, що зберінається в колонці 23  Unnamed: 22*

In [5]:
# Перевіряємо унікальні значення в колонці
print("Значення в колонці Unnamed: 22:")
print(amazon_df['Unnamed: 22'].value_counts(dropna=False))

Значення в колонці Unnamed: 22:
Unnamed: 22
False    79925
NaN      49050
Name: count, dtype: int64


🧹✨ **План очищення даних основного датасету**:
- Формат дат: перетворення в datetime.

- Пропуски в Amount: не видаляляємо ці рядки, щоб не втратити інформацію про обсяги замовлень (Qty) та їхні статуси (Cancelled/Shipped), але замінемо NaN на 0, щоб коректно рахувати загальний дохід.

- Дублікати: Це критично для точності. Якщо один і той самий запис порахувати двічі, це викривить фінансові показники.

- Видалимо колонку Unnamed, яка не несе нам користі в аналізі.

In [6]:
import duckdb

# Видаляємо непотрібні колонки
amazon_df.drop(columns=['index', 'Unnamed: 22'], errors='ignore', inplace=True)

# Конвертація дати
amazon_df['Date'] = pd.to_datetime(amazon_df['Date'], errors='coerce')

# Обробка Amount: замінюємо NaN на 0 (оскільки замовлення без суми зазвичай скасовані)
amazon_df['Amount'] = amazon_df['Amount'].fillna(0)

# Видалення дублікатів
amazon_df.drop_duplicates(inplace=True)

# Типізація поштових кодів
amazon_df['ship-postal-code'] = amazon_df['ship-postal-code'].fillna(0).astype(int).astype(str)

print(f"Очищення завершено. Рядків залишилось: {len(amazon_df)}")
amazon_df.head(3)

/tmp/ipykernel_1845/2897841447.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon_df['Date'] = pd.to_datetime(amazon_df['Date'], errors='coerce')


Очищення завершено. Рядків залишилось: 128969


,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,...,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,0,INR,647.62,MUMBAI,MAHARASHTRA,400081,IN,NaN,False,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,1,INR,406.00,BENGALURU,KARNATAKA,560085,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN


🧹✨ **Аналіз та очищення products_df**

In [7]:
# Подивимося на структуру та пропуски в таблиці товарів
print("--- Інформація про products_df ---")
print(products_df.info())

print("\n--- Пропуски в розрізі колонок ---")
print(products_df.isnull().sum())

# Подивимося на перші 5 рядків
products_df.head(5)

--- Інформація про products_df ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9271 entries, 0 to 9270
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   index       9271 non-null   int64  
 1   SKU Code    9188 non-null   object 
 2   Design No.  9235 non-null   object 
 3   Stock       9235 non-null   float64
 4   Category    9226 non-null   object 
 5   Size        9235 non-null   object 
 6   Color       9226 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 507.1+ KB
None

--- Пропуски в розрізі колонок ---
index          0
SKU Code      83
Design No.    36
Stock         36
Category      45
Size          36
Color         45
dtype: int64


,index,SKU Code,Design No.,Stock,Category,Size,Color
0,0,AN201-RED-L,AN201,5.0,AN : LEGGINGS,L,Red
1,1,AN201-RED-M,AN201,5.0,AN : LEGGINGS,M,Red
2,2,AN201-RED-S,AN201,3.0,AN : LEGGINGS,S,Red
3,3,AN201-RED-XL,AN201,6.0,AN : LEGGINGS,XL,Red
4,4,AN201-RED-XXL,AN201,3.0,AN : LEGGINGS,XXL,Red


In [8]:
# Видаляємо пропуски в SKU, бо це наш ключ для об'єднання
products_df.dropna(subset=['SKU Code'], inplace=True)

# Перейменовуємо колонку для консистентності з основною таблицею
products_df.rename(columns={'SKU Code': 'SKU'}, inplace=True)

# Видаляємо непотрібну колонку index
if 'index' in products_df.columns:
    products_df.drop(columns=['index'], inplace=True)

# Видаляємо повні дублікати рядків
products_df.drop_duplicates(inplace=True)

# Видаляємо дублікати за SKU
products_df.drop_duplicates(subset=['SKU'], keep='first', inplace=True)

# Заповнюємо пропуски в інших колонках та робимо приведення типів
products_df['Stock'] = products_df['Stock'].fillna(0).astype(int)
products_df['Category'] = products_df['Category'].fillna('Unknown')
products_df['Size'] = products_df['Size'].fillna('Unknown')
products_df['Color'] = products_df['Color'].fillna('Unknown')

print(f"Очищення завершено. Унікальних товарів у довіднику: {len(products_df)}")
products_df.head()

Очищення завершено. Унікальних товарів у довіднику: 9170


,SKU,Design No.,Stock,Category,Size,Color
0,AN201-RED-L,AN201,5,AN : LEGGINGS,L,Red
1,AN201-RED-M,AN201,5,AN : LEGGINGS,M,Red
2,AN201-RED-S,AN201,3,AN : LEGGINGS,S,Red
3,AN201-RED-XL,AN201,6,AN : LEGGINGS,XL,Red
4,AN201-RED-XXL,AN201,3,AN : LEGGINGS,XXL,Red


🧹✨ Додавання, аналіз та очищення таблиці Profit&Loss, яку ми також використаємо в подальшому аналізі

In [9]:
# Завантажуємо таблицю з цінами на різних маркетплейсах
pl_df = pd.read_csv('/content/project_data/P  L March 2021.csv')

print("--- Початковий огляд таблиці цін ---")
print(pl_df.info())

pl_df.head()

--- Початковий огляд таблиці цін ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1330 entries, 0 to 1329
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   index           1330 non-null   int64 
 1   Sku             1330 non-null   object
 2   Style Id        1330 non-null   object
 3   Catalog         1330 non-null   object
 4   Category        1330 non-null   object
 5   Weight          1330 non-null   object
 6   TP 1            1330 non-null   object
 7   TP 2            1330 non-null   object
 8   MRP Old         1330 non-null   object
 9   Final MRP Old   1330 non-null   object
 10  Ajio MRP        1330 non-null   object
 11  Amazon MRP      1330 non-null   object
 12  Amazon FBA MRP  1330 non-null   object
 13  Flipkart MRP    1330 non-null   object
 14  Limeroad MRP    1330 non-null   object
 15  Myntra MRP      1330 non-null   object
 16  Paytm MRP       1330 non-null   object
 17  Snapdeal MRP   

,index,Sku,Style Id,Catalog,Category,Weight,TP 1,TP 2,MRP Old,Final MRP Old,Ajio MRP,Amazon MRP,Amazon FBA MRP,Flipkart MRP,Limeroad MRP,Myntra MRP,Paytm MRP,Snapdeal MRP
0,0,Os206_3141_S,Os206_3141,Moments,Kurta,0.3,538,435.78,2178,2295,2295,2295,2295,2295,2295,2295,2295,2295
1,1,Os206_3141_M,Os206_3141,Moments,Kurta,0.3,538,435.78,2178,2295,2295,2295,2295,2295,2295,2295,2295,2295
2,2,Os206_3141_L,Os206_3141,Moments,Kurta,0.3,538,435.78,2178,2295,2295,2295,2295,2295,2295,2295,2295,2295
3,3,Os206_3141_XL,Os206_3141,Moments,Kurta,0.3,538,435.78,2178,2295,2295,2295,2295,2295,2295,2295,2295,2295
4,4,Os206_3141_2XL,Os206_3141,Moments,Kurta,0.3,538,435.78,2178,2295,2295,2295,2295,2295,2295,2295,2295,2295


In [10]:
# Перейменовуємо 'Sku' на 'SKU' для синхронізації з іншими таблицями
pl_df.rename(columns={'Sku': 'SKU'}, inplace=True)

# Список колонок, які мають бути числовими
cols_to_fix = [
    'Weight', 'TP 1', 'TP 2', 'MRP Old', 'Final MRP Old',
    'Ajio MRP', 'Amazon MRP', 'Amazon FBA MRP',
    'Flipkart MRP', 'Limeroad MRP', 'Myntra MRP', 'Paytm MRP', 'Snapdeal MRP'
]

# Очищення: замінюємо 'Nill' на NaN та конвертуємо в числа
for col in cols_to_fix:
    if col in pl_df.columns:
        # Спочатку замінюємо текст 'Nill' на пусті значення
        pl_df[col] = pl_df[col].replace('Nill', pd.NA)
        # Конвертуємо в числа, ігноруючи помилки (текст стане NaN)
        pl_df[col] = pd.to_numeric(pl_df[col], errors='coerce')

# Видаляємо дублікати за SKU (нам потрібна одна ціна для кожного товару)
pl_df.drop_duplicates(subset=['SKU'], keep='first', inplace=True)

# Видаляємо технічну колонку index
if 'index' in pl_df.columns:
    pl_df.drop(columns=['index'], inplace=True)

print("--- Очищення завершено ---")
print(pl_df.info())
pl_df.head()

--- Очищення завершено ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1330 entries, 0 to 1329
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   SKU             1330 non-null   object 
 1   Style Id        1330 non-null   object 
 2   Catalog         1330 non-null   object 
 3   Category        1330 non-null   object 
 4   Weight          1257 non-null   float64
 5   TP 1            1324 non-null   float64
 6   TP 2            1324 non-null   float64
 7   MRP Old         1293 non-null   float64
 8   Final MRP Old   1293 non-null   float64
 9   Ajio MRP        1293 non-null   float64
 10  Amazon MRP      1293 non-null   float64
 11  Amazon FBA MRP  1293 non-null   float64
 12  Flipkart MRP    1293 non-null   float64
 13  Limeroad MRP    1293 non-null   float64
 14  Myntra MRP      1299 non-null   float64
 15  Paytm MRP       1293 non-null   float64
 16  Snapdeal MRP    1293 non-null   float64
dtypes: flo

,SKU,Style Id,Catalog,Category,Weight,TP 1,TP 2,MRP Old,Final MRP Old,Ajio MRP,Amazon MRP,Amazon FBA MRP,Flipkart MRP,Limeroad MRP,Myntra MRP,Paytm MRP,Snapdeal MRP
0,Os206_3141_S,Os206_3141,Moments,Kurta,0.3,538.0,435.78,2178.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0
1,Os206_3141_M,Os206_3141,Moments,Kurta,0.3,538.0,435.78,2178.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0
2,Os206_3141_L,Os206_3141,Moments,Kurta,0.3,538.0,435.78,2178.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0
3,Os206_3141_XL,Os206_3141,Moments,Kurta,0.3,538.0,435.78,2178.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0
4,Os206_3141_2XL,Os206_3141,Moments,Kurta,0.3,538.0,435.78,2178.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0,2295.0


### **Об'єднання трьох таблиць**
Використаємо один SQL-запит, який з'єднає основні продажі з описом товарів та цінами конкурентів ( це також скоротить нам код, оскількт при використанні pandas це зайняло б більше ресурсів.

In [11]:
import duckdb

# SQL-запит
sql_query = """
SELECT
    a.*,
    p.Color,
    p.Stock,
    pl."Ajio MRP",
    pl."Flipkart MRP",
    pl."Myntra MRP"
FROM amazon_df AS a
LEFT JOIN products_df AS p ON a.SKU = p.SKU
LEFT JOIN pl_df AS pl ON a.SKU = pl.SKU
"""

# Виконуємо запит і зберігаємо результат у final_df
final_df = duckdb.query(sql_query).to_df()

print(f"✅ Об'єднання через SQL зроблено")
print(f"Розмір фінальної таблиці: {final_df.shape}")

# Перевіряємо результат
final_df[['Order ID', 'SKU', 'Amount', 'Color', 'Flipkart MRP']].head()

✅ Об'єднання через SQL зроблено
Розмір фінальної таблиці: (128969, 27)


,Order ID,SKU,Amount,Color,Flipkart MRP
0,404-3674282-8853902,SET375-KR-NP-XXXL,654.0,Maroon,NaN
1,404-0017324-4386702,MEN5011-KR-M,688.0,Blue,NaN
2,403-9104047-6761908,JNE3458-KR-XXXL,369.0,Brown,NaN
3,403-0361224-6434742,J0277-SKD-XL,1323.0,Gold,NaN
4,408-1481880-6219535,J0003-SET-XS,664.0,Mustard,NaN


**Перевірка цілісності даних** (Data Validation)
Перевіряємо, чи не виникло помилок при SQL-об'єднанні.
Важливо переконатися, що LEFT JOIN не створив зайвих рядків (дублікатів) і що дані з довідників підтягнулися коректно.

In [12]:
# Перевіряємо чи збігається кількість рядків з оригіналом
print(f"Кількість замовлень в оригіналі: {len(amazon_df)}")
print(f"Кількість замовлень після об'єднання: {len(final_df)}")

# Перевіряємо чи підтягнулися дані з інших таблиць (Color та ціни)
missing_info = final_df[['Color', 'Flipkart MRP']].isnull().sum()
print("\nКількість замовлень без додаткової інформації:")
print(missing_info)

Кількість замовлень в оригіналі: 128969
Кількість замовлень після об'єднання: 128969

Кількість замовлень без додаткової інформації:
Color             7706
Flipkart MRP    128969
dtype: int64


**Створення зведеної таблиці**:

In [13]:
# Pivot table: Category x Status (count) + cancellation rate
pivot_cat_status = pd.pivot_table(
    final_df,
    index="Category",
    columns="Status",
    values="Order ID",
    aggfunc="count",
    fill_value=0
)

pivot_cat_status["Total_Orders"] = pivot_cat_status.sum(axis=1)

if "Cancelled" in pivot_cat_status.columns:
    pivot_cat_status["Cancel_Rate"] = (pivot_cat_status["Cancelled"] / pivot_cat_status["Total_Orders"]).round(4)
else:
    pivot_cat_status["Cancel_Rate"] = 0

pivot_cat_status = pivot_cat_status.sort_values("Cancel_Rate", ascending=False)

display(pivot_cat_status.head(15))

Status,Cancelled,Pending,Pending - Waiting for Pick Up,Shipped,Shipped - Damaged,Shipped - Delivered to Buyer,Shipped - Lost in Transit,Shipped - Out for Delivery,Shipped - Picked Up,Shipped - Rejected by Buyer,Shipped - Returned to Seller,Shipped - Returning to Seller,Shipping,Total_Orders,Cancel_Rate
Category,,,,,,,,,,,,,,,
Set,7336,252,108,30663,0,10644,2,19,406,6,766,73,6,50281,0.1459
kurta,7253,248,72,30793,0,10451,2,5,297,2,715,35,1,49874,0.1454
Western Dress,2122,92,79,7480,1,5190,1,6,186,1,314,28,0,15500,0.1369
Bottom,60,1,2,222,0,139,0,0,10,0,5,1,0,440,0.1364
Saree,21,0,0,119,0,22,0,0,1,0,1,0,0,164,0.1280
Blouse,116,3,1,623,0,169,0,0,2,0,12,0,0,926,0.1253
Ethnic Dress,145,7,0,755,0,234,0,0,2,0,16,0,0,1159,0.1251
Top,1276,55,19,7143,0,1920,0,5,69,2,124,8,1,10622,0.1201
Dupatta,0,0,0,3,0,0,0,0,0,0,0,0,0,3,0.0000


### ❓ Питання №1: Динаміка доходів у часі (Time Series Analysis)
*Як змінювався щоденний дохід (Revenue) протягом усього періоду?*

Це ключова бізнес-метрика. Вона дозволяє власнику магазину побачити піки продажів (наприклад, свята чи акції) та загальний тренд розвитку.

In [14]:
import plotly.express as px

# Агрегуємо дані - групуємо по даті та сумуємо Amount
daily_sales = final_df.groupby('Date')['Amount'].sum().reset_index()

# Побудуємо графік
fig = px.line(daily_sales,
              x='Date',
              y='Amount',
              title='📈 Щоденна динаміка доходу на Amazon',
              labels={'Amount': 'Дохід (INR)', 'Date': 'Дата'},
              template='plotly_white')

# Додаємо лінію тренду (7-денне середнє), щоб краще бачити динаміку
daily_sales['Rolling_Mean'] = daily_sales['Amount'].rolling(window=7).mean()
fig.add_scatter(x=daily_sales['Date'], y=daily_sales['Rolling_Mean'], name='Тренд (7 днів)', line=dict(dash='dash'))

fig.show()

🧐 **Ми агрегували дані про продажі за датами та побудували інтерактивний лінійний графік з додаванням 7-денного середнього ковзного (тренд)**.

**Аналіз результатів**:

- Наявність піку активності (кінець квітня -початок травня): Найвищий рівень доходу спостерігається в період з 24 квітня по 8 травня, де денний виторг перевищує 1.1 млн - 1.2 млн. Це свідчить про проведення великої акції (наприклад, сезонного розпродажу) або святковий період, що стимулював попит.

- Загальний тренд: пунктирна лінія тренду показує, що після піку в травні спостерігається поступове зниження обсягів продажів до кінця червня. Це може бути пов’язано з вичерпанням попиту після акцій або сезонним переходом асортименту.

- Висока волатильність: синя лінія показує різкі щоденні коливання. Це підтверджує, що в e-commerce продажі сильно залежать від конкретного дня тижня та короткострокових рекламних пушів.

Наші рекомендації:

- Необхідно проаналізувати, які саме категорії товарів спричинили пік у травні, щоб повторити цей успіх у майбутньому.

- Зниження тренду в червні потребує впровадження нових маркетингових активностей для підтримки рівня доходу.

### **❓ Питання №2: Аналіз ефективності категорій (Category Performance)**
*Наша мета*: Визначити, які категорії товарів приносять основну частку доходу, а які є менш ефективними.


*Бізнес-цінність*: Знаючи найприбутковіші категорії, компанія може краще планувати закупівлі та розподіляти рекламний бюджет.

In [15]:
# Використовуємо SQL для агрегації доходу за категоріями
category_revenue = duckdb.query("""
    SELECT
        Category,
        SUM(Amount) as Total_Revenue,
        COUNT("Order ID") as Order_Count
    FROM final_df
    WHERE Status NOT IN ('Cancelled', 'Rejected') -- аналізуємо тільки успішні транзакції
    GROUP BY Category
    ORDER BY Total_Revenue DESC
""").to_df()

# Побудова інтерактивної стовпчикової діаграми
import plotly.express as px

fig = px.bar(category_revenue,
             x='Category',
             y='Total_Revenue',
             title='💰 Розподіл доходу за категоріями товарів',
             labels={'Total_Revenue': 'Загальний дохід (INR)', 'Category': 'Категорія'},
             color='Total_Revenue',
             text_auto='.2s', # відображає суми над стовпчиками (напр. 45M)
             color_continuous_scale='Viridis')

# Налаштування вигляду
fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig.show()

🧐 **Ми використали SQL-запит для агрегації сум продажів за категоріями товарів (відфільтрувавши скасовані замовлення) та побудували інтерактивну стовпчикову діаграму (Bar Chart)**.

**Аналіз результатів**:

- Абсолютний лідер цн "Set": Категорія "Set" генерує левову частку доходу - 36 млн INR. Це більше, ніж дохід від трьох наступних категорій разом узятих. Це свідчить про те, що комплекти одягу є найбільш затребуваним продуктом у покупців.

- Основні гравці: категорії "kurta" (19 млн INR) та "Western Dress" (10 млн INR) також демонструють високі показники. Разом із "Set" ці три категорії формують основу фінансової стабільності магазину.

- Низькоприбуткові категорії: такі категорії, як "Saree", "Bottom" та особливо "Dupatta" (лише 920 INR), приносять мінімальний дохід. Це може бути ознакою того, що ці товари не є цільовими для твого магазину або потребують перегляду маркетингової стратегії.

**Наші рекомендації**:

- Фокус на запаси: оскільки "Set" та "kurta" приносять найбільше грошей, необхідно забезпечити постійну наявність цих товарів на складі (уникати Out of Stock).

- Оптимізація маркетингу: рекламні кампанії доцільно спрямовувати саме на лідерів продажу для максимізації прибутку.

- Перегляд асортименту: варто задуматися, чи доцільно витрачати ресурси на підтримку категорії "Dupatta" та "Saree", якщо їхній внесок у загальний дохід є незначним.

### ❓ **Питання 3. Який метод виконання замовлень (Fulfilment) є домінуючим та ефективнішим**?

*Мета*: Порівняти кількість замовлень та обсяг доходу між двома моделями: Amazon (Fulfillment by Amazon - цн коли логістикою займається склад маркетплейса) та Merchant (Fulfillment by Merchant - коли продавець відправляє товар сам).

*Навіщо це бізнесу*:

- Масштабованість: якщо більшість замовлень йде через Amazon, бізнес менше залежить від власних складів.

- Довіра: Товари з позначкою "Fulfilled by Amazon" зазвичай мають вищу довіру покупців і швидшу доставку.

In [16]:
# SQL-запит для аналізу логістики
fulfillment_stats = duckdb.query("""
    SELECT
        Fulfilment,
        COUNT("Order ID") as Order_Count,
        SUM(Amount) as Total_Revenue
    FROM final_df
    GROUP BY Fulfilment
""").to_df()

# Будуємо інтерактивну секторну діаграму (Pie Chart)
fig = px.pie(fulfillment_stats,
             values='Order_Count',
             names='Fulfilment',
             title='🚚 Частка методів виконання замовлень (за кількістю)',
             hole=0.4, # Робимо діаграму у формі кільця (Donut chart)
             color_discrete_sequence=['#FF9900', '#232F3E']) # додамо фірмові кольори Amazon

fig.update_traces(textinfo='percent+label', pull=[0.1, 0]) # трохи "витягнемо" один сегмент
fig.show()

# Додамо короткий звіт по цифрах під графіком
print("Статистика виконання замовлень:")
display(fulfillment_stats)

Статистика виконання замовлень:


,Fulfilment,Order_Count,Total_Revenue
0,Amazon,89692,54319516.0
1,Merchant,39277,24270527.3


🧐 **Аналіз результатів:**

- Домінування логістики Amazon: як видно з діаграми, 69.5% усіх замовлень (89692 шт.) виконуються через систему Amazon. Це свідчить про високу довіру продавця до інфраструктури маркетплейса.

- Частка власної доставки: 30.5% замовлень (39277 шт.) продавець доставляє самостійно (Merchant).

*Модель Amazon приносить понад 54.3 млн INR доходу, тоді як Merchant — близько 24.3 млн INR. Це підтверджує, що товари, які знаходяться на складах Amazon, купують значно частіше та на більші суми.*

💡 Бізнес-інсайти:
- Масштабованість: використання FBA (Amazon) дозволяє бізнесу обробляти величезну кількість замовлень без розширення власного штату складських працівників.

- Лояльність покупців: зазвичай товари з позначкою "Fulfilled by Amazon" мають вищий пріоритет у видачі та доставляються швидше, що пояснює такий високий відсоток замовлень у цій групі.

- Ризики: Висока залежність від Amazon (майже 70%) робить бізнес вразливим до зміни комісій маркетплейса. Варто підтримувати модель Merchant як надійний альтернативний канал.

### ❓ **Питання №4  Аналіз скасувань та повернень** (Cancellation Analysis)
Ми знаємо, скільки замовлень ми робимо, але скільки з них реально доходять до грошей? Високий рівень скасувань — це "тихий вбивця" прибутку.

*Мета*: З'ясувати, чи є певні категорії товарів або методи доставки (Amazon vs Merchant), де скасування трапляються найчастіше.

Кожне скасоване замовлення - це втрачені гроші на логістиці та маркетингу. Розуміння того, де саме «відпадають» клієнти, дозволяє зменшити ці втрати (Churn Rate)

In [17]:
# SQL-запит для розрахунку кількості успішних та скасованих замовлень по категоріях
cancel_analysis = duckdb.query("""
    SELECT
        Category,
        CASE WHEN Status = 'Cancelled' THEN 'Cancelled' ELSE 'Successful' END as Order_Status,
        COUNT("Order ID") as Order_Count
    FROM final_df
    GROUP BY Category, Order_Status
""").to_df()

# Побудова групованої стовпчикової діаграми
fig = px.bar(cancel_analysis,
             x='Category',
             y='Order_Count',
             color='Order_Status',
             barmode='group',
             title='📊 Порівняння успішних та скасованих замовлень за категоріями',
             labels={'Order_Count': 'Кількість замовлень', 'Category': 'Категорія', 'Order_Status': 'Статус'},
             color_discrete_map={'Successful': '#2ECC71', 'Cancelled': '#E74C3C'}) # хелений для успіху, червоний для скасувань

fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Додатковий розрахунок відсотка скасувань для кожної категорії (метрика Churn Rate)
pivot_df = cancel_analysis.pivot(index='Category', columns='Order_Status', values='Order_Count').fillna(0)
pivot_df['Cancel_Rate_%'] = (pivot_df['Cancelled'] / (pivot_df['Cancelled'] + pivot_df['Successful']) * 100).round(2)

print("Топ-5 категорій з найвищим відсотком скасувань:")
display(pivot_df.sort_values(by='Cancel_Rate_%', ascending=False).head(5))

Топ-5 категорій з найвищим відсотком скасувань:


Order_Status,Cancelled,Successful,Cancel_Rate_%
Category,,,
Set,7336.0,42945.0,14.59
kurta,7253.0,42621.0,14.54
Western Dress,2122.0,13378.0,13.69
Bottom,60.0,380.0,13.64
Saree,21.0,143.0,12.80


🧐 **Аналіз результатів**:

- Рівномірність відмов: відсоток скасувань у топ-категоріях дуже схожий і коливається в межах 13.5% – 14.6%. Це свідчить про те, що проблема скасувань є системною для всього магазину, а не притаманна лише одній групі товарів.

- Масштаб втрат у лідерів: Категорії "Set" (14.59%) та "kurta" (14.54%) мають не лише найбільший дохід, а й найбільшу кількість скасувань в абсолютних цифрах — понад 7300 та 7200 замовлень відповідно. Кожне таке скасування - це витрати на логістику та пакування, які не окупилися.

- Критичні зони: Навіть у категорій з меншим обсягом продажів, таких як "Bottom" (13.64%) та "Saree" (12.80%), рівень скасувань залишається високим. Це сигнал, що покупці часто передумують або стикаються з проблемами на етапі доставки.

💡 Бізнес-рекомендації:
- Аналіз причин: Оскільки відсоток скасувань перевищує 14% у лідерів, варто провести опитування клієнтів або проаналізувати коментарі. Часто причинами в Індії є занадто довга доставка або невідповідність розмірній сітці.

- Покращення описів: Високий рівень скасувань "Western Dress" (13.69%) може бути пов'язаний з тим, що фото товару не повністю відповідає реальності.

- Економічна ефективність: Скорочення рівня скасувань лише на 2-3% для категорії "Set" може зберегти бізнесу сотні тисяч рупій.

### ❓ **Питання №5. Географічний аналіз: де знаходяться наші ключові ринки**?
*Мета*: Визначити топ-10 штатів за обсягом продажів та візуалізувати розподіл замовлень по країні.

*Навіщо це бізнесу*:

- Логістика: якщо ми бачимо концентрацію продажів у певному регіоні, ми можемо перенести туди основні запаси, щоб скоротити час доставки.

- Маркетинг:  ми можемо налаштувати таргетовану рекламу саме на ті штати, де попит найвищий.

In [18]:
# SQL-запит для агрегації продажів по штатах
state_sales = duckdb.query("""
    SELECT
        "ship-state" as State,
        SUM(Amount) as Total_Revenue,
        COUNT("Order ID") as Order_Count
    FROM final_df
    WHERE Status != 'Cancelled'
    GROUP BY State
    ORDER BY Total_Revenue DESC
    LIMIT 10
""").to_df()

# Візуалізація: Топ-10 штатів (Горизонтальний Bar Chart)
# Ми використовуємо горизонтальний формат, щоб назви штатів було зручно читати
fig = px.bar(state_sales,
             y='State',
             x='Total_Revenue',
             orientation='h',
             title='🏆 Топ-10 штатів за обсягом доходу (INR)',
             labels={'Total_Revenue': 'Загальний дохід', 'State': 'Штат'},
             color='Total_Revenue',
             text_auto='.2s',
             color_continuous_scale='Portland')

fig.update_layout(yaxis={'categoryorder':'total ascending'}) # Найкращий штат буде зверху
fig.show()

# Висновок по штатах
print("Статистика по ключових регіонах:")
display(state_sales)

Статистика по ключових регіонах:


,State,Total_Revenue,Order_Count
0,MAHARASHTRA,12223831.0,19293
1,KARNATAKA,9649981.0,15081
2,TELANGANA,6290128.0,9696
3,UTTAR PRADESH,6186123.0,9033
4,TAMIL NADU,5954349.0,9889
5,DELHI,3906089.0,5894
6,KERALA,3377321.0,5410
7,WEST BENGAL,3207213.0,5079
8,ANDHRA PRADESH,2882206.0,4538
9,HARYANA,2654274.0,3852


### ❓ **Питання №6. Статистична перевірка: чи є суттєва різниця між B2B та B2C клієнтами?**

*Чому це важливо?*

У бізнесі часто здається, що одна група клієнтів (наприклад, гуртові B2B) приносить більше грошей за раз, ніж інші (звичайні B2C). Але чи це справді так, чи це просто випадковість у даних? Статистичний тест (T-test) може надати нам обгрунтовані дані для відповіді.

In [19]:
import scipy.stats as stats
import plotly.express as px

# Підготовка даних
# вибираємо тільки успішні замовлення (Amount > 0) для обох груп
b2b_sales = final_df[(final_df['B2B'] == True) & (final_df['Amount'] > 0)]['Amount']
b2c_sales = final_df[(final_df['B2B'] == False) & (final_df['Amount'] > 0)]['Amount']

# Проводимо розрахунок основних показників
mean_b2b = b2b_sales.mean()
mean_b2c = b2c_sales.mean()

# Проведення Т-тесту (порівняння середніх значень двох груп)
t_stat, p_value = stats.ttest_ind(b2b_sales, b2c_sales, equal_var=False)

print(f"--- Статистичні показники ---")
print(f"Середній чек B2B: {mean_b2b:.2f} INR")
print(f"Середній чек B2C: {mean_b2c:.2f} INR")
print(f"Різниця: {abs(mean_b2b - mean_b2c):.2f} INR")
print(f"P-value: {p_value:.10f}") # виводимо з багатьма знаками після коми

# Візуалізація: Box Plot (Ящик з вусами)
fig = px.box(final_df[final_df['Amount'] > 0],
             x='B2B',
             y='Amount',
             color='B2B',
             title='📦 Порівняння розподілу сум замовлень: B2B vs B2C',
             labels={'Amount': 'Сума замовлення (INR)', 'B2B': 'Тип клієнта (True = B2B)'},
             points=False, # ее показуємо всі тисячі точок, щоб не гальмував Colab
             notched=True) # додає виїмку на медіані для візуального порівняння

fig.show()

--- Статистичні показники ---
Середній чек B2B: 701.33 INR
Середній чек B2C: 661.06 INR
Різниця: 40.27 INR
P-value: 0.0005259161


**🧐 Аналіз**:

- Математичне підтвердження: оскільки отримане значення $P-value$ значно менше за стандартний поріг у 0.05, ми відхиляємо нульову гіпотезу про рівність середніх чеків. Це означає, що різниця у 40.27 INR не є випадковою помилкою вибірки, а є реальною закономірністю поведінки клієнтів.

- Профіль клієнта: Бізнес-клієнти (B2B) у середньому витрачають на замовлення на 6% більше, ніж звичайні покупці. Це підтверджує гіпотезу про те, що гуртові або корпоративні замовлення мають вищу цінність для бізнесу.

- Щодо візуалізація розподілу (Box Plot): хоча «коробки» на графіку частково перекриваються, медіана B2B знаходиться вище, ніж у B2C. Також ми бачимо, що розмах значень (вуса графіка) в обох групах сягає понад 5000 INR, що свідчить про наявність дорогих преміальних замовлень у обох сегментах.

💡 Бізнес-рекомендації:
- Пріоритезація B2B сегменту: попри те, що B2B клієнтів може бути менше за кількістю, їхня вища прибутковість за одну транзакцію виправдовує розробку спеціальних умов: персональних менеджерів або накопичувальних систем знижок для корпоративних замовлень.

- Маркетингова стратегія: варто налаштувати окремі рекламні кампанії на LinkedIn або через професійні мережі, оскільки залучення одного B2B клієнта приносить компанії більше прибутку, ніж залучення звичайного покупця.

### ❓ **Питання 7. Machine Learning: Чи можемо ми передбачити скасування замовлення**?

*Наша мета*: побудувати модель, яка на основі даних про замовлення (категорія, сума, спосіб доставки тощо) зможе визначити ймовірність того, що клієнт його скасує.

*Навіщо це бізнесу*:

Якщо система автоматично позначає замовлення як "ризиковане" (з високою ймовірністю скасування), менеджер може зателефонувати клієнту для підтвердження або запропонувати бонус, щоб утримати його. Це допомагає економити на логістиці "вхолосту".

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Підготовка даних
# вибираємо фактори, які можуть впливати на скасування
features = ['Category', 'Fulfilment', 'Qty', 'Amount', 'B2B']
X = final_df[features].copy()

# створюємо цільову змінну: 1 якщо скасовано, 0 якщо успішно
y = (final_df['Status'] == 'Cancelled').astype(int)

# Перетворюємо текстові категорії в числа (One-Hot Encoding)
X = pd.get_dummies(X, columns=['Category', 'Fulfilment'], drop_first=True)

# 2. Розділяємо дані на навчальну та тестову вибірки (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Дані підготовлено. Навчальна вибірка: {X_train.shape[0]} записів.")

# Створюємо та навчаємо модель
# використовуємо 100 дерев для балансу швидкості та якості
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Оцінюємо модель на тестових даних
y_pred = rf_model.predict(X_test)

print("\n--- Звіт про ефективність моделі (Classification Report) ---")
print(classification_report(y_test, y_pred))

# Визначаємо найважливіші фактори
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nТоп-5 факторів, що найбільше впливають на прогноз:")
print(importances.head(5))

Дані підготовлено. Навчальна вибірка: 103175 записів.

--- Звіт про ефективність моделі (Classification Report) ---
              precision    recall  f1-score   support

           0       0.95      1.00      0.98     22128
           1       0.99      0.71      0.82      3666

    accuracy                           0.96     25794
   macro avg       0.97      0.85      0.90     25794
weighted avg       0.96      0.96      0.95     25794


Топ-5 факторів, що найбільше впливають на прогноз:
                   feature  importance
0                      Qty    0.798749
1                   Amount    0.190609
11     Fulfilment_Merchant    0.009671
2                      B2B    0.000230
9   Category_Western Dress    0.000180


🧐 **Результати моделювання**:

- Загальна точність (Accuracy): 96%. Це означає, що у 96 випадках зі 100 модель правильно вгадує долю замовлення.

- Якість прогнозу скасувань (Class 1): precision (Точність) = 0.99: Якщо модель каже, що замовлення буде скасовано, вона права у 99% випадків.

- Recall (Повнота) = 0.71: модель самостійно знаходить 71% усіх реальних скасувань. Це дуже хороший результат для базової моделі.

*Головні чинники впливу* (Feature Importance):

- кількість (Qty) — 80% впливу:  виявляється, саме кількість одиниць у замовленні є вирішальним фактором. Можливо, замовлення з великою кількістю або нульовою кількістю (як ми бачили раніше) автоматично ведуть до скасування.

- Сума (Amount) — 19% впливу: Ціна замовлення також грає роль. Дорожчі замовлення мають іншу ймовірність скасування, ніж дешеві.

- Спосіб доставки (Fulfilment_Merchant): має невеликий, але помітний вплив (близько 1%).

*Бізнес-цінність*:
Завдяки цій моделі бізнес може автоматично виділяти замовлення з високим ризиком скасування (де Qty та Amount виходять за межі норми) і перевіряти їх вручну до того, як витрачати гроші на логістику.

### ❓ **Питання №8 Які штати мають найвищий cancel rate (але тільки серед топ-штатів за обсягом)**?
*Мета*: Знайти географічні “гарячі точки” скасувань серед штатів з достатнім обсягом замовлень.

*Для чого це бізнесу*:
- Де вище Cancelled там проблеми логістики/термінів/оплати/повернень у конкретних регіонах.
- Дає рішення: міняти партнерів доставки, SLA, комунікацію для певних регіонів.

In [29]:
df = final_df.copy()

state_col = "ship-state" if "ship-state" in df.columns else "State"

state_kpi = (
    df.groupby(state_col)
    .agg(
        Orders=("Order ID", "count"),
        Cancel_Rate=("Status", lambda s: (s == "Cancelled").mean()),
        Total_Revenue=("Amount", "sum")
    )
    .reset_index()
)

state_kpi["Cancel_Rate"] = (state_kpi["Cancel_Rate"] * 100).round(2)
state_kpi["Total_Revenue"] = state_kpi["Total_Revenue"].round(2)

# фільтр: беремо тільки штати з достатнім N
min_orders = 50
state_kpi_f = state_kpi[state_kpi["Orders"] >= min_orders].sort_values("Cancel_Rate", ascending=False)

display(state_kpi_f.head(10))

fig = px.scatter(
    state_kpi_f,
    x="Orders",
    y="Cancel_Rate",
    hover_name=state_col,
    size="Total_Revenue",
    title=f"Зв'язок між обсягом замовлень та cancel rate по штатах (Orders ≥ {min_orders})",
    labels={"Orders": "К-сть замовлень", "Cancel_Rate": "Скасування, %"}
)
fig.show()


,ship-state,Orders,Cancel_Rate,Total_Revenue
31,MIZORAM,75,18.67,41213.71
20,HIMACHAL PRADESH,788,18.53,503364.51
24,KERALA,6584,17.83,3830227.58
0,ANDAMAN & NICOBAR,257,17.51,158723.62
13,DADRA AND NAGAR,70,17.14,42138.92
21,JAMMU & KASHMIR,702,16.95,456932.74
1,ANDHRA PRADESH,5430,16.43,3219831.72
8,BIHAR,2086,16.30,1394388.32
39,ODISHA,2115,16.12,1372205.63
38,New Delhi,81,16.05,47109.95


🧐  **Аналіз**: найвищий cancel rate мають Mizoram (18.67%) та Himachal Pradesh (18.53%), однак Mizoram має низьку кількість замовлень (75), тому результат може бути нестабільним. Kerala (17.83%) поєднує високий cancel rate з великим обсягом (6584), отже є пріоритетом для глибшого аналізу причин скасувань. Рекомендовано сфокусуватися на регіонах із високим cancel rate та достатнім обсягом (Orders ≥ 50/100) і перевірити логістичні SLA, терміни доставки та часті причини відміни замовлень у цих штатах.

### ❓  **Питання №9  Як кількість товарів у замовленні (Qty) впливає на cancel rate?**

*Мета* : оцінити, чи великі кошики частіше скасовуються.

*Для чого це бізнесу*: якщо для Qty ≥ 4/6 скасування різко зростає — варто додати підтвердження/контроль/умови оплати для великих замовлень.

In [30]:
bins = [-np.inf, 1, 2, 3, 5, np.inf]
labels = ["1", "2", "3", "4-5", "6+"]

df["Qty_Group"] = pd.cut(df["Qty"], bins=bins, labels=labels)

qty_kpi = (
    df.groupby("Qty_Group")
    .agg(
        Orders=("Order ID", "count"),
        Avg_Check=("Amount", "mean"),
        Cancel_Rate=("Status", lambda s: (s == "Cancelled").mean())
    )
    .reset_index()
)

qty_kpi["Avg_Check"] = qty_kpi["Avg_Check"].round(2)
qty_kpi["Cancel_Rate"] = (qty_kpi["Cancel_Rate"] * 100).round(2)

display(qty_kpi)

fig = px.line(
    qty_kpi,
    x="Qty_Group",
    y="Cancel_Rate",
    markers=True,
    title="Питання №9: Частка скасувань (%) залежно від кількості товарів у замовленні (Qty)",
    labels={"Qty_Group": "Група Qty", "Cancel_Rate": "Скасування, %"}
)
fig.show()

/tmp/ipykernel_1845/2240321780.py:7: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,Qty_Group,Orders,Avg_Check,Cancel_Rate
0,1,128581,607.40,14.23
1,2,341,1197.81,7.62
2,3,32,1558.22,0.00
3,4-5,11,2332.18,0.00
4,6+,4,1396.00,0.00


Аналіз: найбільша частка скасувань у замовлень з Qty=1 (14.23%). Для Qty≥3 cancel rate дорівнює 0%, але ці групи мають дуже малу кількість замовлень (N=32/11/4), тому висновок про залежність для великих кошиків є ненадійним. Для коректної інтерпретації варто агрегувати рідкісні групи або застосувати мінімальний поріг N.

In [33]:
df = final_df.copy()

# прибираємо некоректні Qty
df = df[df["Qty"] > 0].copy()

# Групи 1 / 2 / 3+
df["Qty_Group2"] = np.where(df["Qty"] >= 3, "3+", df["Qty"].astype(int).astype(str))

qty_kpi2 = (
    df.groupby("Qty_Group2", observed=True)
    .agg(
        Orders=("Order ID", "count"),
        Avg_Check=("Amount", "mean"),
        Cancel_Rate=("Status", lambda s: (s == "Cancelled").mean())
    )
    .reset_index()
)

qty_kpi2["Avg_Check"] = qty_kpi2["Avg_Check"].round(2)
qty_kpi2["Cancel_Rate"] = (qty_kpi2["Cancel_Rate"] * 100).round(2)

display(qty_kpi2)

fig = px.bar(
    qty_kpi2,
    x="Qty_Group2",
    y="Cancel_Rate",
    title="Питання №9 (оновлено): Частка скасувань (%) для Qty=1, Qty=2, Qty=3+ (Qty>0)",
    labels={"Qty_Group2": "Група Qty", "Cancel_Rate": "Скасування, %"},
    text="Cancel_Rate"
)
fig.show()

,Qty_Group2,Orders,Avg_Check,Cancel_Rate
0,1,115777,647.02,4.84
1,2,341,1197.81,7.62
2,3+,47,1725.55,0.00


**Аналіз**:

після фільтрації некоректних записів Qty=0 (аналізуємо тільки Qty>0) видно, що основна маса замовлень припадає на Qty=1 (115 777 замовлень) із cancel rate 4.84%. Для Qty=2 cancel rate вищий — 7.62%, але вибірка дуже мала (341), тому висновок потребує обережної інтерпретації. Група Qty=3+ має лише 47 замовлень і 0% скасувань, що не дозволяє робити надійні висновки про великі кошики.


### ❓  **Питання 10 Який ship-service-level має найвищий cancel rate і який дає найбільший обсяг замовлень/виручки**

*Мета* : Порівняти рівні доставки за часткою скасувань, щоб визначити “проблемний” сервіс і зрозуміти вплив на бізнес..

*Для чого це бізнесу*: високий cancel rate у певному сервісі → втрати доходу + навантаження на підтримку.

In [34]:

service_col = "ship-service-level"

service_kpi = (
    df.groupby(service_col)
    .agg(
        Orders=("Order ID", "count"),
        Total_Revenue=("Amount", "sum"),
        Avg_Check=("Amount", "mean"),
        Cancel_Rate=("Status", lambda s: (s == "Cancelled").mean())
    )
    .reset_index()
)

service_kpi["Total_Revenue"] = service_kpi["Total_Revenue"].round(2)
service_kpi["Avg_Check"] = service_kpi["Avg_Check"].round(2)
service_kpi["Cancel_Rate"] = (service_kpi["Cancel_Rate"] * 100).round(2)

display(service_kpi.sort_values("Orders", ascending=False))

fig = px.bar(
    service_kpi,
    x=service_col,
    y="Cancel_Rate",
    title="Питання №10: Частка скасувань (%) за рівнем доставки (ship-service-level)",
    labels={service_col: "Рівень доставки", "Cancel_Rate": "Скасування, %"},
    text="Cancel_Rate"
)
fig.show()

,ship-service-level,Orders,Total_Revenue,Avg_Check,Cancel_Rate
0,Expedited,82720,54282548.0,656.22,6.80
1,Standard,33445,21117323.0,631.40,0.01


Аналіз:

- рівень доставки Expedited має суттєво вищу частку скасувань 6.80% (82 720 замовлень), тоді як Standard майже не має скасувань 0.01% (33 445 замовлень).

- При цьому Expedited генерує більшу частину виручки (~54.3M) порівняно зі Standard (~21.1M), тому саме Expedited найбільше впливає на втрати доходу через скасування.
- Також середній чек схожий (656.22 vs 631.40), отже різниця в скасуваннях, ймовірно, пов’язана не з ціною, а з процесом виконання/доставки або очікуваннями клієнта щодо швидкості.

Рекомендації бізнесу:
- Пріоритезувати аналіз причин скасувань саме для Expedited (SLA, затримки, доступність товару на складі, помилки в обіцянці термінів)
- Перевірити, на якому етапі стаються скасування Expedited (до відвантаження чи вже в процесі доставки) - це допоможе вибрати правильний важіль: склад/логістика/комунікація

### **ЗАГАЛЬНІ ВИСНОВКИ**:

Основна цінність проведеного аналізу полягає у виявленні реальних драйверів прибутку та критичних точок втрат. Ми з'ясували, що **асортимент тримається на категоріях "Set" та "Kurta", які формують понад 70% виторгу**, а **географічним серцем продажів є штат Maharashtra**. Статистичне дослідження підтвердило, що **B2B-сегмент є більш прибутковим** ($p < 0.05$), оскільки демонструє вищий середній чек порівняно з роздрібними покупцями. Це дає чіткий сигнал маркетингу: інвестиції в утримання одного корпоративного клієнта приносять більше дивідендів, ніж масове охоплення B2C.**Найбільшим викликом для бізнесу є операційна ефективність логістики**. Ми виявили аномально високий відсоток скасувань у замовленнях типу Expedited (6.80%), що призводить до подвійних логістичних витрат. Проте розроблена модель машинного навчання (Random Forest) з точністю 96% дозволяє прогнозувати ці скасування ще на етапі оформлення. Для бізнесу це означає перехід від "гасіння пожеж" до превентивного управління: можливість додатково перевіряти ризикові замовлення перед відправкою, зберігаючи обігові кошти та товар у наявності.

### 💡 **Ключові рекомендації**:

- Інтеграція ML-рішення: впровадити розроблену модель прогнозування у внутрішню CRM-систему для автоматичного маркування "ризикових" замовлень. Це дозволить менеджерам зв'язуватися з клієнтом для підтвердження перед відправкою.

- Оптимізація B2B-стратегії: розробити спеціальну програму лояльності або систему знижок для корпоративних клієнтів, оскільки вони є найбільш стабільним та високочековим сегментом.

- Ревізія швидкої доставки: провести аудит процесів Expedited доставки. Високий відсоток скасувань вказує на розрив між очікуваннями клієнта та реальністю (можливо, через затримки або неактуальні залишки на складі).

- Фокус на маржинальність: використовуючи результати кореляційного аналізу, змістити акцент у рекламі на легкі, але дорогі товари. Це дозволить максимізувати прибуток, мінімізуючи витрати на логістику, які напряму залежать від ваги.